# CineScale - Colab ETL & ALS Training Pipeline

This notebook extracts ratings and movies, cleans the data, trains an ALS
model, evaluates it, and exports the embeddings to Parquet.

Unlike the Databricks serverless version, this runs local Spark with full
SparkContext access, so checkpointing and broadcast variables work normally.

## 0. Setup

Run this cell first. It installs PySpark and starts a local Spark session.

In [1]:
import warnings
warnings.filterwarnings("ignore")
!pip install -q pyspark

from pyspark.sql import SparkSession

sprk = (
    SparkSession.builder
    .appName("CineScale")
    .master("local[*]")
    .config("spark.driver.memory", "6g") # Adjusted driver memory to 6g
    .config("spark.executor.memory", "6g") # Explicitly setting executor memory
    .config("spark.sql.shuffle.partitions", "200") # was 100
    .config("spark.default.parallelism", "200")
    .getOrCreate()
)

# Full SparkContext access is available on Colab, unlike Databricks serverless.
# This lets ALS checkpoint properly during training, which prevents the
# lineage-explosion problem that caused the 12+ hour hang on serverless.
sprk.sparkContext.setCheckpointDir("/content/checkpoints")

print("Spark version:", sprk.version)

Spark version: 4.0.3


## 1. Get your data onto Colab

Two options — uncomment whichever you're using.

**Option A: Upload directly** (good for smaller files, e.g. MovieLens 1M/10M)

In [2]:
# from google.colab import files
# uploaded = files.upload()  # select movies.csv and ratings.csv.gz from the file picker

**Option B: Mount Google Drive** (better for larger files you'll reuse)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

Point this at wherever your files ended up (Colab's local filesystem if you
used the upload cell, or a Drive path if you mounted Drive).

In [4]:
import logging
from pyspark.sql.functions import col, trim, explode
from pyspark.sql.types import IntegerType, StringType, FloatType, LongType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# If you used the upload cell, files land in /content/
# If you mounted Drive, point this at your Drive folder instead, e.g.:
# RAW_DATA_DIR = "/content/drive/MyDrive/cinescale"
RAW_DATA_DIR = "/content/drive/MyDrive/CineScale"
PROCESSED_DATA_DIR = "/content/drive/MyDrive/CineScale/procced"
EMBEDDING_DIM = 50

## 3. ETL (Extract, Transform, Load)

In [5]:
import os
print(f"Contents of RAW_DATA_DIR ({RAW_DATA_DIR}):")
if os.path.exists(RAW_DATA_DIR):
    for item in os.listdir(RAW_DATA_DIR):
        print(item)
else:
    print(f"Directory does not exist: {RAW_DATA_DIR}")

Contents of RAW_DATA_DIR (/content/drive/MyDrive/CineScale):
ratings.csv
movies.csv
procced


In [6]:
def extract_data(raw_dir: str):
    movies_path = f"{raw_dir}/movies.csv"
    ratings_path = f"{raw_dir}/ratings.csv"

    print(f"Loading raw movies from {movies_path}")
    movies_df = sprk.read.option("header", "true").csv(movies_path)

    print(f"Loading raw ratings from {ratings_path}")
    ratings_df = sprk.read.option("header", "true").csv(ratings_path)

    return movies_df, ratings_df

def transform_movies(movies_df):
    filtered_df = movies_df.filter(col("movieId").isNotNull())
    filtered_df = filtered_df.filter(trim(col("movieId")).rlike(r"^\d+$"))
    casted_df = filtered_df \
        .withColumn("movieId", trim(col("movieId")).cast(IntegerType())) \
        .withColumn("title", col("title").cast(StringType())) \
        .withColumn("genres", col("genres").cast(StringType()))
    transformed_df = casted_df.fillna("Unknown", subset=["title", "genres"])
    return transformed_df

def transform_ratings(ratings_df):
    print(f"Raw ratings count in transform_ratings: {ratings_df.count()}")
    filtered_df_not_null = ratings_df.filter(
        col("userId").isNotNull() & col("movieId").isNotNull() & col("rating").isNotNull()
    )
    print(f"Count after not null filter: {filtered_df_not_null.count()}")

    filtered_df_regex = filtered_df_not_null.filter(
        trim(col("userId")).rlike(r"^\d+$") &
        trim(col("movieId")).rlike(r"^\d+$") &
        trim(col("rating")).rlike(r"^\d+(\.\d+)?$")
    )
    print(f"Count after regex filter: {filtered_df_regex.count()}")

    transformed_df = filtered_df_regex \
        .withColumn("userId", trim(col("userId")).cast(IntegerType())) \
        .withColumn("movieId", trim(col("movieId")).cast(IntegerType())) \
        .withColumn("rating", trim(col("rating")).cast(FloatType())) \
        .withColumn("timestamp", trim(col("timestamp")).cast(LongType()))

    print("Checking for nulls after casting (sampling 10 rows):")
    transformed_df.select(
        col("userId").isNull().alias("userId_is_null"),
        col("movieId").isNull().alias("movieId_is_null"),
        col("rating").isNull().alias("rating_is_null"),
        col("timestamp").isNull().alias("timestamp_is_null")
    ).limit(10).show()

    return transformed_df

def filter_low_support_items(ratings_df, min_ratings=10):
    item_counts = ratings_df.groupBy("movieId").count()
    valid_items = item_counts.filter(col("count") >= min_ratings).select("movieId")
    filtered_df = ratings_df.join(valid_items, on="movieId", how="inner")
    print(f"Ratings before: {ratings_df.count()}, after min_ratings={min_ratings} filter: {filtered_df.count()}")
    return filtered_df

# Execute ETL
raw_movies_df, raw_ratings_df = extract_data(RAW_DATA_DIR)
clean_movies_df = transform_movies(raw_movies_df)
clean_ratings_df = transform_ratings(raw_ratings_df)
clean_ratings_df = filter_low_support_items(clean_ratings_df, min_ratings=10)

# Cache is fully supported here (unlike Databricks Free Edition serverless).
# This also materializes the ETL result once, instead of it being recomputed
# every time it's touched downstream.
clean_ratings_df.cache()
print(f"Clean ratings row count: {clean_ratings_df.count()}")

Loading raw movies from /content/drive/MyDrive/CineScale/movies.csv
Loading raw ratings from /content/drive/MyDrive/CineScale/ratings.csv
Raw ratings count in transform_ratings: 32000204
Count after not null filter: 32000204
Count after regex filter: 32000204
Checking for nulls after casting (sampling 10 rows):
+--------------+---------------+--------------+-----------------+
|userId_is_null|movieId_is_null|rating_is_null|timestamp_is_null|
+--------------+---------------+--------------+-----------------+
|         false|          false|         false|            false|
|         false|          false|         false|            false|
|         false|          false|         false|            false|
|         false|          false|         false|            false|
|         false|          false|         false|            false|
|         false|          false|         false|            false|
|         false|          false|         false|            false|
|         false|          f

### Diagnose empty clean_ratings_df

Let's inspect the raw ratings DataFrame to understand why all rows are being dropped during the cleaning process. We'll look at the first few rows and the schema.

In [7]:
print('First 5 rows of raw_ratings_df:')
raw_ratings_df.show(5)

print('Schema of raw_ratings_df:')
raw_ratings_df.printSchema()

print('First 5 rows of clean_ratings_df to confirm it is empty:')
clean_ratings_df.show(5)


First 5 rows of raw_ratings_df:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
|     1|     30|   5.0|944249077|
|     1|     32|   5.0|943228858|
+------+-------+------+---------+
only showing top 5 rows
Schema of raw_ratings_df:
root
 |-- userId: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- timestamp: string (nullable = true)

First 5 rows of clean_ratings_df to confirm it is empty:
+-------+------+------+----------+
|movieId|userId|rating| timestamp|
+-------+------+------+----------+
|    148|  1278|   3.0| 838407261|
|    148|  1695|   4.0| 831240840|
|    148|  1735|   0.5|1234676991|
|    148|  2811|   2.0| 833173771|
|    148|  3408|   3.0| 940857361|
+-------+------+------+----------+
only showing top 5 rows


## 4. Train-Test Split

In [8]:
def split_data(ratings_df, train_ratio=0.8, seed=42):
    return ratings_df.randomSplit([train_ratio, 1 - train_ratio], seed=seed)

train_df, test_df = split_data(clean_ratings_df)
train_df.cache()
test_df.cache()

print(f"Train ratings: {train_df.count()}, Test ratings: {test_df.count()}")

Train ratings: 25470280, Test ratings: 6372425


## 5. ALS Model Training

`checkpointInterval=5` truncates the join lineage every 5 iterations, using
the checkpoint dir set up in step 0. This is what was silently unavailable
on Databricks serverless and caused the 12+ hour hang.

In [11]:
def train_als_model(train_df):
    als = ALS(
    rank=50,
    maxIter=25,
    regParam=0.05,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    implicitPrefs=True,
    alpha=40,
    coldStartStrategy="drop",
    checkpointInterval=5,
    seed=42,
)
    return als.fit(train_df)

print(f"Training ALS model (rank={EMBEDDING_DIM})")
model = train_als_model(train_df)

# Cache and materialize the factors once, right after training, so nothing
# downstream ever replays the ALS iteration chain.
model.userFactors.cache()
model.itemFactors.cache()
model.userFactors.count()
model.itemFactors.count()

Training ALS model (rank=50)


31960

In [12]:
sample_user = test_df.select("userId").distinct().first()["userId"]

# What the model recommends
uf = model.userFactors.filter(col("id") == sample_user).collect()[0]["features"]
itf = model.itemFactors.toPandas()
import numpy as np
scores = np.array(itf["features"].tolist()) @ np.array(uf)
top10_idx = np.argsort(-scores)[:10]
top10_movies = itf["id"].values[top10_idx]
print("Top-10 recommended:", top10_movies)

# What the user actually liked in test
relevant = test_df.filter((col("userId") == sample_user) & (col("rating") >= 4.0)) \
                   .select("movieId").rdd.flatMap(lambda r: r).collect()
print("Test relevant items:", relevant)

Top-10 recommended: [2205 5300 4704 5499 4802 4767 6448 4401 4795 6405]
Test relevant items: [897, 3986, 3105, 961, 1653, 2294, 2003, 3198, 2355, 2240, 2467, 3359, 30793, 4240, 2236, 5266, 1197, 3104, 3798, 48394, 1302, 3147, 49688, 1722, 13, 2471, 1242, 4993, 25795, 1301, 6032, 7019, 1028, 2014, 46578, 8039, 48, 959, 2085, 3039, 44709, 4534, 32116, 1950, 1396, 1124, 2819, 2968, 48982, 5945, 1017, 3408, 8529, 2875, 2297, 6539, 2204, 1022, 1254, 380, 434, 277, 2077, 7099, 10, 5506, 8783, 53125, 527, 4395, 5378, 6565, 6593, 6232, 351, 480, 55895, 1610, 7075, 2870, 911, 828, 8636, 44204, 4835, 6808, 786, 52287, 457, 4306, 5604, 5952, 7619, 3559, 1196, 913, 2916, 6986, 5299, 56171, 2028, 1672, 1693, 158, 2762, 1262, 1373, 1012, 4085, 7579, 1376, 1674, 3524, 1713, 8644, 932, 6428, 5530, 7482, 1387]


## 6. Evaluation (RMSE & Precision@10)

In [13]:
def compute_rmse(model, test_df):
    predictions = model.transform(test_df)
    evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
    return evaluator.evaluate(predictions)

import gc
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, IntegerType


def compute_precision_at_k(
    model,
    train_df,
    test_df,
    spark,
    k: int = 10,
    rating_threshold: float = 4.0,
    user_batch_size: int = 2000,
    arrow_batch_size: int = 500,
    user_col: str = "user",
    item_col: str = "item",
    rating_col: str = "rating",
):
    spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", str(arrow_batch_size))

    print("1. Collecting Item Factors to driver...")
    item_factors_pdf = model.itemFactors.toPandas()
    item_ids = item_factors_pdf["id"].values.astype(np.int32)
    item_matrix = np.vstack(item_factors_pdf["features"].values).astype(np.float32)
    print(f"   Items: {len(item_ids)}, item_matrix memory: {item_matrix.nbytes / (1024**2):.2f} MB")
    del item_factors_pdf
    gc.collect()

    print("2. Broadcasting item data...")
    item_ids_bc = spark.sparkContext.broadcast(item_ids)
    item_matrix_bc = spark.sparkContext.broadcast(item_matrix)

    # FIX #1: Much larger buffer so filtering training items never empties the list
    K_FETCH_BUFFER = 5000

    @pandas_udf(ArrayType(IntegerType()))
    def get_top_k_recs(user_features_series: pd.Series) -> pd.Series:
        items = item_ids_bc.value
        i_matrix = item_matrix_bc.value
        user_matrix = np.vstack(user_features_series.values).astype(i_matrix.dtype)
        all_scores = user_matrix @ i_matrix.T

        K_fetch = min(k + K_FETCH_BUFFER, len(items))
        result = []
        for scores in all_scores:
            top_idx = np.argpartition(scores, -K_fetch)[-K_fetch:]
            top_idx = top_idx[np.argsort(-scores[top_idx])]
            result.append(items[top_idx].tolist())

        del all_scores, user_matrix
        return pd.Series(result)

    print("3. Collecting distinct test users...")
    test_users = [r[user_col] for r in test_df.select(user_col).distinct().collect()]
    print(f"   {len(test_users)} distinct test users -> batching by {user_batch_size}")

    print("3a. Preparing relevant items per user (test set, rating >= threshold)...")
    model_item_ids_df = model.itemFactors.select(col("id").alias(item_col)).distinct()

    relevant_items_df = (
        test_df.filter(col(rating_col) >= rating_threshold)
        .join(model_item_ids_df, on=item_col, how="inner")
        .groupBy(user_col)
        .agg(F.collect_set(item_col).alias("relevant_items"))
        .filter(F.size("relevant_items") > 0)
    )
    relevant_items_df.cache()
    n_relevant_users = relevant_items_df.count()
    print(f"   Number of users with model-known relevant items: {n_relevant_users}")

    print("3b. Preparing each user's training-set items for exclusion...")
    train_items_df = (
        train_df.groupBy(user_col)
        .agg(F.collect_set(item_col).alias("train_items"))
    )
    train_items_df.cache()
    train_items_df.count()

    total_precision_sum = 0.0
    total_hits_sum = 0.0
    total_recall_sum = 0.0
    total_users_counted = 0
    n_batches = (len(test_users) + user_batch_size - 1) // user_batch_size

    for b in range(n_batches):
        batch_users = test_users[b * user_batch_size : (b + 1) * user_batch_size]
        print(f"   Batch {b + 1}/{n_batches} - {len(batch_users)} users")

        user_batch_df = (
            model.userFactors.filter(col("id").isin(batch_users))
            .withColumnRenamed("id", user_col)
        )

        recs_df = user_batch_df.withColumn(
            "recommendations", get_top_k_recs(col("features"))
        ).select(user_col, "recommendations")

        joined = (
            recs_df.join(relevant_items_df, on=user_col, how="inner")
            .join(train_items_df, on=user_col, how="left")
        )

        metrics_row = joined.select(
            F.expr(
                f"size(array_intersect("
                f"slice(array_except(recommendations, coalesce(train_items, array())), 1, {k}), "
                f"relevant_items)) / {k}"
            ).alias("precision"),
            F.expr(
                f"IF(size(array_intersect("
                f"slice(array_except(recommendations, coalesce(train_items, array())), 1, {k}), "
                f"relevant_items)) > 0, 1.0, 0.0)"
            ).alias("hit"),
            F.expr(
                f"size(array_intersect("
                f"slice(array_except(recommendations, coalesce(train_items, array())), 1, {k}), "
                f"relevant_items)) / size(relevant_items)"
            ).alias("recall"),
        )

        agg = metrics_row.agg(
            F.sum("precision").alias("sum_p"),
            F.sum("hit").alias("sum_hit"),
            F.sum("recall").alias("sum_r"),
            F.count("*").alias("cnt"),
        ).collect()[0]

        if agg["cnt"]:
            total_precision_sum += float(agg["sum_p"])
            total_hits_sum += float(agg["sum_hit"])
            total_recall_sum += float(agg["sum_r"])
            total_users_counted += agg["cnt"]

        user_batch_df.unpersist()
        recs_df.unpersist()
        del user_batch_df, recs_df, joined, metrics_row
        gc.collect()

    item_ids_bc.unpersist()
    item_matrix_bc.unpersist()
    relevant_items_df.unpersist()
    train_items_df.unpersist()

    avg_precision = total_precision_sum / total_users_counted if total_users_counted else 0.0
    avg_hit_rate = total_hits_sum / total_users_counted if total_users_counted else 0.0
    avg_recall = total_recall_sum / total_users_counted if total_users_counted else 0.0

    print(f"Precision@{k}: {avg_precision:.4f}")
    print(f"Hit Rate@{k}: {avg_hit_rate:.4f}")
    print(f"Recall@{k}: {avg_recall:.4f}")
    print(f"(over {total_users_counted} users)")

    return {"precision": avg_precision, "hit_rate": avg_hit_rate, "recall": avg_recall}


# ---------------------------------------------------------------------------
# If you're still hitting OOM after batching, the JVM heap itself is probably
# too small. Set this BEFORE creating your SparkSession (can't be changed on
# a running session) -- e.g. in the Colab cell that builds `spark`/`sprk`:
#
# from pyspark.sql import SparkSession
# spark = (
#     SparkSession.builder
#     .config("spark.driver.memory", "10g")        # give the JVM real headroom
#     .config("spark.driver.maxResultSize", "4g")
#     .config("spark.sql.execution.arrow.maxRecordsPerBatch", "500")
#     .getOrCreate()
# )
#
# Then call:
# compute_precision_at_k(model, train_df, test_df, spark, k=10,
#                         user_batch_size=2000, arrow_batch_size=500)
# ---------------------------------------------------------------------------


print("Computing RMSE on test set...")
rmse = compute_rmse(model, test_df)
print(f"RMSE: {rmse:.4f}")

print("Computing Precision@10 on test set...")
metrics = compute_precision_at_k(
    model, train_df, test_df, sprk, k=10, rating_threshold=4.0,
    user_col="userId", item_col="movieId", rating_col="rating"
)
print(f"Precision@10: {metrics['precision']:.4f}")
print(f"Hit Rate@10:   {metrics['hit_rate']:.4f}")
print(f"Recall@10:     {metrics['recall']:.4f}")

Computing RMSE on test set...
RMSE: 2.8449
Computing Precision@10 on test set...
1. Collecting Item Factors to driver...
   Items: 31960, item_matrix memory: 6.10 MB
2. Broadcasting item data...
3. Collecting distinct test users...
   200741 distinct test users -> batching by 2000
3a. Preparing relevant items per user (test set, rating >= threshold)...
   Number of users with model-known relevant items: 196658
3b. Preparing each user's training-set items for exclusion...
   Batch 1/101 - 2000 users
   Batch 2/101 - 2000 users
   Batch 3/101 - 2000 users
   Batch 4/101 - 2000 users
   Batch 5/101 - 2000 users
   Batch 6/101 - 2000 users
   Batch 7/101 - 2000 users
   Batch 8/101 - 2000 users
   Batch 9/101 - 2000 users
   Batch 10/101 - 2000 users
   Batch 11/101 - 2000 users
   Batch 12/101 - 2000 users
   Batch 13/101 - 2000 users
   Batch 14/101 - 2000 users
   Batch 15/101 - 2000 users
   Batch 16/101 - 2000 users
   Batch 17/101 - 2000 users
   Batch 18/101 - 2000 users
   Batch 19

## 7. Export Embeddings to Parquet

In [14]:
import os
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Extract User Factors
user_factors = model.userFactors.select(
    col("id").alias("userId"),
    col("features").alias("embedding")
)

# Extract Movie Factors
movie_factors = model.itemFactors.select(
    col("id").alias("movieId"),
    col("features").alias("embedding")
)

print(f"Writing user factors to {PROCESSED_DATA_DIR}/user_factors.parquet")
user_factors.write.mode("overwrite").parquet(f"{PROCESSED_DATA_DIR}/user_factors.parquet")

print(f"Writing movie factors to {PROCESSED_DATA_DIR}/movie_factors.parquet")
movie_factors.write.mode("overwrite").parquet(f"{PROCESSED_DATA_DIR}/movie_factors.parquet")

print("Training pipeline complete!")

Writing user factors to /content/drive/MyDrive/CineScale/procced/user_factors.parquet
Writing movie factors to /content/drive/MyDrive/CineScale/procced/movie_factors.parquet
Training pipeline complete!
